> **Note**: This notebook requires:
> - `data/derived/model_input.csv` — output of `02_prepare_model_input.ipynb`
> - `data/raw/video-positions/` — per-frame 3D centroid files (from `01_video_processing.ipynb`)
> 
> Output:
> - `data/derived/model_input_time_dependent.csv` — model input with time-dependent neighbour distances

# S01 — Calculate time-dependent neighbour distances

For each focal fish and each influencing neighbour, this notebook calculates the 3D distance between them *at the frame the neighbour responded* (rather than a static distance). This is used in the supplementary analysis to verify that using stationary distances — as done in the simulations — does not meaningfully change model results.

## Load packages and paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = ROOT

ALIGN_FRAME = 20  # frame at which the first responder (FR) is aligned across videos
SMD = 5           # sensory-motor delay (frames)

## Define distance calculation function

In [ ]:
def calculate_distance(focal_id, neighbor_id, focal_frame, neighbor_frame, traj_df):
    """Calculate 3D Euclidean distance between two fish at specified frames."""
    focal_pos = traj_df[
        (traj_df['Fish_ID'] == focal_id) &
        (traj_df['Frame_ID'] == focal_frame)
    ][['X', 'Y', 'Z']]
    neighbor_pos = traj_df[
        (traj_df['Fish_ID'] == neighbor_id) &
        (traj_df['Frame_ID'] == neighbor_frame)
    ][['X', 'Y', 'Z']]

    if focal_pos.empty or neighbor_pos.empty:
        return np.nan

    return np.linalg.norm(focal_pos.values[0] - neighbor_pos.values[0])

## Calculate time-dependent distances

In [ ]:
base = pd.read_csv(f'{path}/data/derived/model_input.csv')
base['Input_Distance_at_InputResponse'] = np.nan

for index, row in base.iterrows():
    if row.get('Influence', 0) != 1:
        continue

    vid = row['Video_ID']
    dep_id = row['Dep_ID']
    focal_id = row['Focal_Fish_ID']
    neighbor_id = row['Input_Fish_ID']

    first_response_frame = int(np.nanmin(base.loc[base['Video_ID'] == vid, 'Input_Responseframe_cam3']))
    input_response_frame = int(row['Input_Responseframe_cam3'])

    traj_path = path / f'data/raw/video-positions/deployment_{dep_id}/video_{vid}/Centroids_Video_Overall_{vid}.csv'
    if not traj_path.exists():
        print(f'[WARN] Trajectory file not found: {traj_path}')
        continue

    traj_df = pd.read_csv(traj_path)

    neighbor_frame = (input_response_frame - first_response_frame) + ALIGN_FRAME
    focal_frame = neighbor_frame + SMD

    base.at[index, 'Input_Distance_at_InputResponse'] = calculate_distance(
        focal_id, neighbor_id, focal_frame, neighbor_frame, traj_df
    )

## Save output

In [ ]:
base.to_csv(f'{path}/data/derived/model_input_time_dependent.csv', index=False)
print(f'Saved {len(base)} rows to data/derived/model_input_time_dependent.csv')